# 06 — GenAI Energy Copilot
**Day 5.** Goal: a working Q&A assistant that answers operator questions using real forecast/health
data plus one clearly-labeled mocked dataset (battery status).

**LLM backend: Groq API** (fast hosted inference for open models, e.g. Llama 3.3). Uses an
OpenAI-compatible chat completions interface via the `groq` Python package.

**Data honesty check, up front:**
- Generation forecast + actuals → **real**, from Day 3/4 models and cleaned SCADA.
- Turbine health ranking → **real**, from the actual 50-turbine alarm/maintenance datasets (SCADA
  telemetry only covers WTG_001, but alarm/maintenance history is genuine fleet-wide data).
- Battery/BESS status → **mocked**. No such dataset exists anywhere in the provided files. The
  copilot is instructed to say so whenever it uses this data — verify that it actually does below.

In [10]:
import sys, os
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # context_builder resolves models/ and data/ relative to cwd

from dotenv import load_dotenv
env_path = PROJECT_ROOT / ".env"
loaded_dotenv = load_dotenv(str(env_path), override=True)
print(f"Loaded .env from {env_path}: {loaded_dotenv}")
print("GROQ_API_KEY present after load:", bool(os.getenv("GROQ_API_KEY")))

from src.utils.config_loader import load_config
from src.agents.context_builder import assemble_full_context, get_turbine_health_ranking, get_mock_battery_status
from src.agents.copilot import ask_copilot, build_system_prompt

cfg = load_config()

if os.getenv("GROQ_API_KEY"):
    print("GROQ_API_KEY found -- copilot will call the real LLM (Groq).")
else:
    print("No GROQ_API_KEY set -- copilot will use the rule-based fallback for every question.")
    print("Copy .env.example to .env and add your key (from console.groq.com) to see real LLM answers.")


Loaded .env from c:\Users\ashok\Downloads\wind-energy-ai-platform\wind-energy-ai-platform\.env: True
GROQ_API_KEY present after load: True
GROQ_API_KEY found -- copilot will call the real LLM (Groq).


## 1. Build the context pipeline
This is the single call that gathers everything: latest actuals, next-day forecasts, real turbine health ranking, and mocked battery status.

In [3]:
context = assemble_full_context(cfg)
print("Generation context:")
for k, v in context["generation"].items():
    print(f"  {k}: {v}")


Generation context:
  anchor_date: 2025-12-31
  turbine_id: WTG_001
  actual_generation_today_kwh: 19733.7
  actual_power_loss_today_kwh: 2528.5
  alarm_minutes_today: 20
  mean_wind_speed_today_mps: 7.85
  seasonal_avg_wind_speed_mps: 7.51
  wind_vs_seasonal_pct: 4.5
  trailing_7d_avg_generation_kwh: 16846.1
  generation_vs_trailing_avg_pct: 17.1
  next_day_generation_forecast_kwh: 16054.1
  next_day_demand_forecast_kwh: None
  demand_is_synthetic_proxy: True
  model_test_mae_kwh: 991.6051401654413


In [4]:
print("Top 5 highest-risk turbines (REAL data, last 90 days):")
pd.DataFrame(context["turbine_health_top5"])


Top 5 highest-risk turbines (REAL data, last 90 days):


,turbine_id,n_failures,total_downtime_hours,max_severity_score,last_failure_date,most_common_component,n_alarms_90d,risk_score
0,WTG_026,53,1763.7,4,2025-12-30 12:00:00,Gearbox,261,315.42
1,WTG_009,50,1690.0,4,2025-12-30 23:00:00,Brake System,261,302.05
2,WTG_032,51,1652.1,4,2025-12-25 22:00:00,Generator,258,300.11
3,WTG_047,52,1567.9,4,2025-12-30 15:00:00,Generator,265,294.04
4,WTG_005,47,1640.9,4,2025-12-30 18:00:00,Brake System,245,290.34


In [5]:
print("Battery status (MOCKED -- no real dataset exists):")
pd.DataFrame(context["battery_status_MOCKED"])


Battery status (MOCKED -- no real dataset exists):


,battery_id,state_of_charge_pct,health_pct,cycle_count,dispatch_priority_score
0,BESS_03,87.2,95.7,1543,83.5
1,BESS_04,78.4,96.0,920,75.3
2,BESS_01,82.6,86.3,1021,71.3
3,BESS_02,64.1,98.7,404,63.3


## 2. Inspect the system prompt actually sent to the LLM
Worth looking at once, so you know exactly what data the model is (and isn't) grounded in.

In [6]:
print(build_system_prompt(context)[:2000])
print("...")


You are the Energy Copilot for an AI-powered wind farm operations platform.
Answer the operator's question using ONLY the data context provided below. Be concise and specific,
citing actual numbers from the context. If something isn't covered by the context, say so plainly
rather than guessing.

IMPORTANT: any data below marked MOCKED is placeholder data for this demo, not a real reading.
If your answer relies on mocked data, say so in the answer (e.g. "based on mocked battery data...").

DATA CONTEXT:
{
  "generation": {
    "anchor_date": "2025-12-31",
    "turbine_id": "WTG_001",
    "actual_generation_today_kwh": 19733.7,
    "actual_power_loss_today_kwh": 2528.5,
    "alarm_minutes_today": 20,
    "mean_wind_speed_today_mps": 7.85,
    "seasonal_avg_wind_speed_mps": 7.51,
    "wind_vs_seasonal_pct": 4.5,
    "trailing_7d_avg_generation_kwh": 16846.1,
    "generation_vs_trailing_avg_pct": 17.1,
    "next_day_generation_forecast_kwh": 16054.1,
    "next_day_demand_forecast_kwh": nul

## 3. Test the four sample operator questions from the work plan
Default model is `llama-3.3-70b-versatile`. If Groq changes model availability and you get a
"model not found" style error in the `error` field below, check
https://console.groq.com/docs/models and pass a different `model=` to `ask_copilot`.

In [7]:
sample_questions = [
    "Why did wind generation drop today?",
    "Predict tomorrow's production.",
    "Which turbine needs maintenance?",
    "Which battery should discharge first?",
]

for q in sample_questions:
    result = ask_copilot(q, context)
    print(f"Q: {q}")
    print(f"[source: {result['source']}" + (f", error: {result['error']}]" if result["error"] else "]"))
    print(f"A: {result['answer']}")
    print("-" * 80)


Q: Why did wind generation drop today?
[source: fallback, error: GROQ_API_KEY not set]
A: On 2025-12-31, WTG_001 generated 19,734 kWh (trailing 7-day average: 16,846 kWh). Generation was actually UP 17.1% vs. the trailing 7-day average -- not a drop. Estimated power loss vs. the expected power curve was 2,528 kWh. Mean wind speed was 7.85 m/s, +4.5% vs. the seasonal average for this week. There were 20 minutes of active alarms that day. [Rule-based fallback answer -- set GROQ_API_KEY for a fuller explanation.]
--------------------------------------------------------------------------------
Q: Predict tomorrow's production.
[source: fallback, error: GROQ_API_KEY not set]
A: Forecast for the day after 2025-12-31: 16,054 kWh generation (model test MAE: 992 kWh). [Rule-based fallback answer.]
--------------------------------------------------------------------------------
Q: Which turbine needs maintenance?
[source: fallback, error: GROQ_API_KEY not set]
A: Highest-risk turbine over the la

## 4. Minimal chat loop
A basic notebook-based chat interface, per the work plan ("no need for a polished UI at this stage").
Run this cell, type a question, type `exit` to stop. Re-run `assemble_full_context(cfg)` first if
you've regenerated any upstream data.

In [1]:
def chat_loop():
    print("Energy Copilot (Groq) -- type 'exit' to stop.\n")
    while True:
        question = input("You: ")
        if question.strip().lower() in ("exit", "quit"):
            print("Copilot: goodbye.")
            break
        result = ask_copilot(question, context)
        tag = "" if result["source"] == "llm" else "  [fallback -- no live LLM call]"
        print(f"Copilot{tag}: {result['answer']}\n")

# Uncomment to run interactively in VS Code / Jupyter:
# chat_loop()


## 5. Documented limitations (per the work plan's Day 5 instruction)

- **Battery data is entirely mocked** — no BESS dataset exists among the provided files. Anything
  the copilot says about battery dispatch is illustrative only.
- **Turbine health is a rule-based risk score**, not a trained failure-probability model (that's
  Report Model 4 / a future phase) — it ranks turbines using real alarm counts and maintenance
  downtime, weighted by simple fixed coefficients, not a learned model.
- **"Today" is simulated** as the last date in the SCADA file (2025-12-31), not a live feed — there
  is no real-time sensor integration in this prototype.
- **Single-turbine grounding**: generation numbers are for WTG_001 only, consistent with the
  dataset's actual coverage (confirmed Day 1–2).
- **Fallback answers are templated**, not LLM-generated — they only cover the four sample questions
  above. Set `GROQ_API_KEY` in `.env` for open-ended Q&A.
- **Hosted model availability is Groq's to change** — `llama-3.3-70b-versatile` is current at time
  of writing; if it's retired, swap the `model=` argument in `ask_copilot`.

State these plainly in the demo — this is exactly the kind of expectation-setting the work plan
calls for.

---
## Checkpoint — Day 5 complete

- `src/agents/context_builder.py` and `src/agents/copilot.py` are reusable by Day 6's multi-agent
  pipeline and Day 8's dashboard — don't rebuild this logic there, import it.
- All four sample questions produce grounded answers, with mocked/real data explicitly labeled.
- Fault tolerance confirmed: the copilot works with or without an API key.

Next: Day 6 — wrap this and the forecasting model into a small set of cooperating agents
(Forecast, Optimization, Maintenance, Grid, Carbon), each logging its decisions in plain language.